# Sesión 6 · Taller: tu primer tablero en Dash
**Misión 9–11 nov 2026 · Miércoles 11 · segunda mitad (~45 min)**

**Encargo:** el gerente de un restaurante quiere un tablero **descriptivo** de sus cuentas y propinas (dataset `tips`, 244 cuentas). Completa las celdas `# TODO` (reemplaza `...`). Sigue la misma estructura de la demo: **funciones → layout → callback → run**.

| Paso | Tema | Min |
|---|---|---|
| 1 | Usuario, pregunta y KPI | 5 |
| 2 | Funciones de KPI y figuras | 12 |
| 3 | Layout con HTML y controles | 10 |
| 4 | Callback | 10 |
| 5 | Ejecutar y probar | 8 |
| ⭐ | Retos | extra |

In [ ]:
# En Google Colab: instalar lo que no viene por defecto (en local puede omitirse)
import importlib.util, subprocess, sys
for pkg in ["dash", "plotly"]:
    if importlib.util.find_spec(pkg.replace('-', '_')) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

In [ ]:
import pandas as pd
import seaborn as sns
import plotly.express as px
from dash import Dash, html, dcc, dash_table, Input, Output

tips = sns.load_dataset("tips")
tips["tip_pct"] = 100 * tips["tip"] / tips["total_bill"]   # % de propina sobre la cuenta
for c in ["sex", "smoker", "day", "time"]:
    tips[c] = tips[c].astype(str)
DIAS = ["Thur", "Fri", "Sat", "Sun"]
tips.head()

In [ ]:
# Estilos reutilizables (CSS escrito como diccionarios de Python)
COLOR = {"fondo": "#F5F7FA", "texto": "#1F2937", "primario": "#1F4E79", "suave": "#6B7280"}
ESTILO_PAGINA = {"fontFamily": "Arial, sans-serif", "backgroundColor": COLOR["fondo"],
                 "padding": "20px", "color": COLOR["texto"]}
ESTILO_TARJETA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "14px 18px",
                  "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "160px",
                  "boxSizing": "border-box"}
ESTILO_FILA = {"display": "flex", "gap": "16px", "flexWrap": "wrap", "marginBottom": "16px"}
ESTILO_CAJA = {"backgroundColor": "white", "borderRadius": "10px", "padding": "8px",
               "boxShadow": "0 1px 4px rgba(0,0,0,.08)", "flex": "1", "minWidth": "380px"}

def tarjeta(titulo, valor, id_valor=None):
    """Tarjeta KPI: título pequeño + valor grande. Con id_valor, el valor se puede actualizar con un callback."""
    props = {"id": id_valor} if id_valor else {}
    return html.Div([
        html.Div(titulo, style={"fontSize": "13px", "color": COLOR["suave"]}),
        html.Div(valor, style={"fontSize": "28px", "fontWeight": "700", "color": COLOR["primario"]}, **props),
    ], style=ESTILO_TARJETA)

## Paso 1 · Usuario, pregunta y KPI (doble clic para editar)

| Usuario | Pregunta | KPI | Gráfico |
|---|---|---|---|
| Gerente | ¿Cuánto vendemos? | Ventas totales (USD) | Barras de ventas por día |
| Gerente | … | % de propina promedio | … |
| Gerente | … | Número de cuentas | — |

## Paso 2 · Funciones de KPI y figuras
Cada función recibe un DataFrame (ya filtrado) y devuelve el resultado.

In [ ]:
def kpis(df):
    if len(df) == 0:
        return {"ventas": "—", "propina": "—", "n": "0"}
    return {"ventas": ...,   # TODO: suma de total_bill, ej. "$4,828"
            "propina": ...,  # TODO: promedio de tip_pct con 1 decimal, ej. "16.1 %"
            "n": ...}        # TODO: número de cuentas como texto

def fig_ventas_dia(df):
    v = df.groupby("day")["total_bill"].sum().reindex(DIAS).fillna(0).reset_index()
    fig = ...  # TODO: px.bar de v con x="day", y="total_bill", text_auto=".0f", labels y un título-hallazgo
    return fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

def fig_propina_dia(df):
    fig = ...  # TODO: px.box de df con x="day", y="tip_pct", category_orders={"day": DIAS}
    return fig.update_layout(margin=dict(t=50, l=10, r=10, b=10))

In [ ]:
# ✅ verificación
k = kpis(tips)
assert k["n"] == "244", k
print(k)
fig_ventas_dia(tips)

## Paso 3 · Layout con HTML y controles
Recuerda: `html.Div([...])` es una caja; con `style=ESTILO_FILA` sus hijos quedan uno al lado del otro.

Debe tener: título (`html.H1`) · fila de controles (`dcc.Checklist` de momento con id `f-momento` y `dcc.Dropdown` de fumador con id `f-fumador`) · fila con 3 tarjetas KPI (ids `k-ventas`, `k-propina`, `k-n`) · fila con los 2 gráficos (ids `g-ventas`, `g-propina`) · `DataTable` (id `tabla`).

In [ ]:
tabla_cols = ["day", "time", "smoker", "size", "total_bill", "tip", "tip_pct"]
app = Dash(__name__)
app.layout = html.Div([
    ...,                                          # TODO: html.H1 con el título del tablero
    html.Div([
        html.Div([html.Label("Momento"),
                  ...], style={"flex": "1"}),     # TODO: dcc.Checklist(id="f-momento", options=["Lunch", "Dinner"], value=[...], inline=True)
        html.Div([html.Label("Mesa de fumadores"),
                  ...], style={"flex": "1"}),     # TODO: dcc.Dropdown(id="f-fumador", options=["Todos", "Yes", "No"], value="Todos")
    ], style={**ESTILO_TARJETA, "display": "flex", "gap": "20px", "marginBottom": "16px"}),
    html.Div([...], style=ESTILO_FILA),           # TODO: 3 tarjetas con tarjeta(titulo, "", id)
    html.Div([...], style=ESTILO_FILA),           # TODO: 2 html.Div(dcc.Graph(id=...), style=ESTILO_CAJA)
    dash_table.DataTable(id="tabla", columns=[{"name": c, "id": c} for c in tabla_cols],
                         page_size=8, sort_action="native"),
], style=ESTILO_PAGINA)

## Paso 4 · Callback
Primero escribe la función `filtrar`, pruébala, y luego conéctala.

In [ ]:
def filtrar(df, momentos, fumador):
    out = ...  # TODO: filas cuyo "time" esté en momentos (usa .isin y `momentos or []`)
    if fumador != "Todos":
        out = ...  # TODO: solo las filas con smoker == fumador
    return out

print(len(filtrar(tips, ["Dinner"], "No")))  # debería dar 106

@app.callback(
    # TODO: 6 Output: los 3 KPI ("children"), los 2 gráficos ("figure") y la tabla ("data")
    # TODO: 2 Input: f-momento y f-fumador ("value")
)
def actualizar(momentos, fumador):
    df = ...    # TODO
    k = ...     # TODO
    return ...  # TODO: en el MISMO orden que los Output

In [ ]:
# ✅ verificación: probar la lógica sin la app
assert len(filtrar(tips, ["Dinner"], "No")) == 106
assert len(filtrar(tips, [], "Todos")) == 0
print("Paso 4 ✔")

## Paso 5 · Ejecutar y probar
Prueba: (1) deja solo "Lunch": ¿qué días desaparecen? (2) compara fumadores vs no fumadores en % de propina. (3) ¿los títulos dicen un hallazgo?

In [ ]:
app.run(jupyter_mode="inline", port=8061, height=1100)

## ⭐ Retos
1. Agrega un `dcc.RangeSlider` para filtrar por tamaño de mesa (`size`, de 1 a 6).
2. Agrega un tercer gráfico: scatter de `total_bill` vs `tip` coloreado por `time`.
3. Copia todo en un archivo `app.py`, agrega `if __name__ == "__main__": app.run(debug=True)` y córrelo con `python app.py`.